In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/retail_clean.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

p1 = df[(df['InvoiceDate'] >= '2009-12-01') & (df['InvoiceDate'] < '2010-12-10')]
p2 = df[(df['InvoiceDate'] >= '2010-12-10') & (df['InvoiceDate'] < '2011-12-10')]

c1 = set(p1['Customer ID'].unique())
c2 = set(p2['Customer ID'].unique())

print(f"Period 1 customers: {len(c1)}")
print(f"Period 2 customers: {len(c2)}")
print(f"In both: {len(c1 & c2)}")
print(f"P1 only: {len(c1 - c2)}")
print(f"P2 only: {len(c2 - c1)}")

Period 1 customers: 4381
Period 2 customers: 4295
In both: 2737
P1 only: 1644
P2 only: 1558


In [2]:
def build_rfm(data, ref_date):
    ref = pd.Timestamp(ref_date)
    g = data.groupby('Customer ID').agg(
        Recency=('InvoiceDate', lambda x: (ref - x.max()).days),
        Frequency=('Invoice', 'nunique'),
        Monetary=('Revenue', 'sum')
    ).reset_index()
    return g[g['Monetary'] > 0]

rfm1 = build_rfm(p1, '2010-12-10')
rfm2 = build_rfm(p2, '2011-12-10')

print(rfm1.shape, rfm2.shape)

(4279, 4) (4247, 4)


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

features = ['Recency', 'Frequency', 'Monetary']

def log_it(d):
    out = d.copy()
    out['Frequency'] = np.log1p(out['Frequency'])
    out['Monetary'] = np.log1p(out['Monetary'])
    return out

l1, l2 = log_it(rfm1), log_it(rfm2)

scaler = StandardScaler().fit(l1[features])
s1 = scaler.transform(l1[features])
s2 = scaler.transform(l2[features])

# Fit K-Means on period 1 only, then predict period 2 against the same centres.
# Fitting separately per period would produce two unrelated labellings.
km = KMeans(n_clusters=4, random_state=42, n_init=10).fit(s1)

rfm1['Cluster'] = km.predict(s1)
rfm2['Cluster'] = km.predict(s2)

rfm1.groupby('Cluster')[features].mean().round(2)

,Recency,Frequency,Monetary
Cluster,,,
0,250.97,1.66,378.88
1,46.42,5.01,1524.97
2,55.13,1.67,381.73
3,20.57,18.96,8690.79


In [4]:
segment_map = {3: 'Champions', 1: 'Loyal Customers', 2: 'New / Low-Engagement', 0: 'Lost'}
rfm1['Segment'] = rfm1['Cluster'].map(segment_map)
rfm2['Segment'] = rfm2['Cluster'].map(segment_map)

merged = rfm1[['Customer ID','Segment']].merge(
    rfm2[['Customer ID','Segment']], on='Customer ID', suffixes=('_p1','_p2')
)
print("Customers in both periods:", len(merged))

order = ['Champions','Loyal Customers','New / Low-Engagement','Lost']
mig = pd.crosstab(merged['Segment_p1'], merged['Segment_p2'], normalize='index').reindex(index=order, columns=order).round(3)
mig

Customers in both periods: 2681


Segment_p2,Champions,Loyal Customers,New / Low-Engagement,Lost
Segment_p1,,,,
Champions,0.561,0.333,0.050,0.055
Loyal Customers,0.105,0.478,0.249,0.168
New / Low-Engagement,0.019,0.258,0.470,0.253
Lost,0.017,0.193,0.368,0.422


In [5]:
vanished = set(rfm1['Customer ID']) - set(rfm2['Customer ID'])
v1 = rfm1[rfm1['Customer ID'].isin(vanished)]
print(v1['Segment'].value_counts())
print("Champion revenue that vanished: £{:,.0f}".format(
    v1[v1['Segment']=='Champions']['Monetary'].sum()))

Segment
New / Low-Engagement    634
Lost                    592
Loyal Customers         334
Champions                38
Name: count, dtype: int64
Champion revenue that vanished: £311,104


In [6]:
mig.to_csv('../data/segment_migration.csv')
print("saved")

saved


In [7]:
excluded = pd.concat([
    rfm1[~rfm1['Customer ID'].isin(merged['Customer ID'])],
    rfm2[~rfm2['Customer ID'].isin(merged['Customer ID'])]
])
retained = pd.concat([
    rfm1[rfm1['Customer ID'].isin(merged['Customer ID'])],
    rfm2[rfm2['Customer ID'].isin(merged['Customer ID'])]
])

print(f"Excluded: {len(excluded)} customer-periods, mean Monetary £{excluded['Monetary'].mean():,.0f}")
print(f"Retained: {len(retained)} customer-periods, mean Monetary £{retained['Monetary'].mean():,.0f}")

Excluded: 3164 customer-periods, mean Monetary £849
Retained: 5362 customer-periods, mean Monetary £2,624
